In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS healthcare_claims_catalog.gold;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ==========================================================
# Read Silver Beneficiary Table
# ==========================================================

beneficiary_df = spark.read.table(
    "healthcare_claims_catalog.silver.beneficiary"
)

# ==========================================================
# Deduplicate to ONE row per patient (FIX)
# silver.beneficiary has 3 rows per DESYNPUF_ID (one per year:
# 2008/2009/2010) by design — that's correct for Silver, but
# DIM_BENEFICIARY must have exactly one row per patient, or every
# fact table joined to it fans out 3x. This keeps the most recent
# year's snapshot per patient.
# ==========================================================

window_spec_dedup = Window.partitionBy("DESYNPUF_ID").orderBy(col("year").desc())

beneficiary_df = beneficiary_df.withColumn(
    "row_num", row_number().over(window_spec_dedup)
).filter(
    col("row_num") == 1
).drop("row_num")

# ==========================================================
# Calculate Age
# ==========================================================

beneficiary_df = beneficiary_df.withColumn(
    "AGE",
    floor(months_between(current_date(), col("BENE_BIRTH_DT")) / 12)
)

# ==========================================================
# Create Age Group
# ==========================================================

beneficiary_df = beneficiary_df.withColumn(
    "AGE_GROUP",
    when(col("AGE") < 18, "0-17")
    .when((col("AGE") >= 18) & (col("AGE") <= 35), "18-35")
    .when((col("AGE") >= 36) & (col("AGE") <= 50), "36-50")
    .when((col("AGE") >= 51) & (col("AGE") <= 64), "51-64")
    .otherwise("65+")
)

# ==========================================================
# Senior Citizen Flag
# ==========================================================

beneficiary_df = beneficiary_df.withColumn(
    "SENIOR_CITIZEN",
    when(col("AGE") >= 65, "Yes")
    .otherwise("No")
)

# ==========================================================
# Gender Description
# ==========================================================

beneficiary_df = beneficiary_df.withColumn(
    "GENDER",
    when(col("BENE_SEX_IDENT_CD") == 1, "Male")
    .when(col("BENE_SEX_IDENT_CD") == 2, "Female")
    .otherwise("Unknown")
)

# ==========================================================
# Race Description
# ==========================================================

beneficiary_df = beneficiary_df.withColumn(
    "RACE",
    when(col("BENE_RACE_CD") == 1, "White")
    .when(col("BENE_RACE_CD") == 2, "Black")
    .when(col("BENE_RACE_CD") == 3, "Other")
    .when(col("BENE_RACE_CD") == 5, "Hispanic")
    .otherwise("Unknown")
)

# ==========================================================
# Generate Surrogate Key
# (now safe — one row per DESYNPUF_ID after the dedup above)
# ==========================================================

window_spec = Window.orderBy("DESYNPUF_ID")

beneficiary_df = beneficiary_df.withColumn(
    "BENEFICIARY_KEY",
    row_number().over(window_spec)
)

# ==========================================================
# Add Gold Audit Column
# ==========================================================

beneficiary_df = beneficiary_df.withColumn(
    "gold_created_timestamp",
    current_timestamp()
)

# ==========================================================
# Select Required Columns
# ==========================================================

dim_beneficiary = beneficiary_df.select(
    "BENEFICIARY_KEY",
    "DESYNPUF_ID",
    "BENE_BIRTH_DT",
    "BENE_DEATH_DT",
    "AGE",
    "AGE_GROUP",
    "SENIOR_CITIZEN",
    "GENDER",
    "RACE",
    "SP_STATE_CODE",
    "SP_ALZHDMTA",
    "SP_CHF",
    "SP_CHRNKIDN",
    "SP_CNCR",
    "SP_COPD",
    "SP_DEPRESSN",
    "SP_DIABETES",
    "SP_ISCHMCHT",
    "SP_OSTEOPRS",
    "SP_RA_OA",
    "SP_STRKETIA",
    "gold_created_timestamp"
)

# ==========================================================
# Write Gold Dimension
# ==========================================================

dim_beneficiary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "healthcare_claims_catalog.gold.dim_beneficiary"
    )

# ==========================================================
# Validation
# ==========================================================

print("=" * 60)
print("Gold Dimension - Beneficiary Created Successfully")
print("=" * 60)

total_records = dim_beneficiary.count()
distinct_patients = dim_beneficiary.select("DESYNPUF_ID").distinct().count()

print("Total Records     :", total_records)
print("Distinct Patients :", distinct_patients)
print("Match (should be True):", total_records == distinct_patients)

dim_beneficiary.printSchema()

dim_beneficiary.show(10, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold Dimension - Beneficiary Created Successfully
Total Records     : 116370
Distinct Patients : 116370
Match (should be True): True
root
 |-- BENEFICIARY_KEY: integer (nullable = false)
 |-- DESYNPUF_ID: string (nullable = true)
 |-- BENE_BIRTH_DT: date (nullable = true)
 |-- BENE_DEATH_DT: date (nullable = true)
 |-- AGE: long (nullable = true)
 |-- AGE_GROUP: string (nullable = false)
 |-- SENIOR_CITIZEN: string (nullable = false)
 |-- GENDER: string (nullable = false)
 |-- RACE: string (nullable = false)
 |-- SP_STATE_CODE: integer (nullable = true)
 |-- SP_ALZHDMTA: integer (nullable = true)
 |-- SP_CHF: integer (nullable = true)
 |-- SP_CHRNKIDN: integer (nullable = true)
 |-- SP_CNCR: integer (nullable = true)
 |-- SP_COPD: integer (nullable = true)
 |-- SP_DEPRESSN: integer (nullable = true)
 |-- SP_DIABETES: integer (nullable = true)
 |-- SP_ISCHMCHT: integer (nullable = true)
 |-- SP_OSTEOPRS: integer (nullable = true)
 |-- SP_RA_OA: integer (nullable = true)
 |-- SP_STRKETIA

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------------+----------------+-------------+-------------+---+---------+--------------+------+-----+-------------+-----------+------+-----------+-------+-------+-----------+-----------+-----------+-----------+--------+-----------+--------------------------+
|BENEFICIARY_KEY|DESYNPUF_ID     |BENE_BIRTH_DT|BENE_DEATH_DT|AGE|AGE_GROUP|SENIOR_CITIZEN|GENDER|RACE |SP_STATE_CODE|SP_ALZHDMTA|SP_CHF|SP_CHRNKIDN|SP_CNCR|SP_COPD|SP_DEPRESSN|SP_DIABETES|SP_ISCHMCHT|SP_OSTEOPRS|SP_RA_OA|SP_STRKETIA|gold_created_timestamp    |
+---------------+----------------+-------------+-------------+---+---------+--------------+------+-----+-------------+-----------+------+-----------+-------+-------+-----------+-----------+-----------+-----------+--------+-----------+--------------------------+
|1              |000002F7E0A96C32|1919-07-01   |NULL         |107|65+      |Yes           |Female|Black|5            |2          |2     |2          |2      |2      |2          |2          |2          |2          |2